In [129]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier


In [130]:
df = pd.read_csv("/Users/shaiv/Documents/log reg/data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [131]:
df = df.drop(columns=['customerID', 'TotalCharges'])

In [132]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [133]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [134]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [135]:
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

/var/folders/rj/mhl1p5l563dgmtt8b3sj9n6w0000gn/T/ipykernel_35714/320056118.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns


In [136]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(drop="first",handle_unknown="ignore"), categorical_cols)
    ]
)

In [137]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=2.8,
    random_state=42
)

In [138]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])


In [139]:
scale_pos_weight = 2.8

In [140]:
param_grid = {

    "model__n_estimators": [200, 400, 600],

    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],

    "model__max_depth": [3, 4, 5, 6, 8],

    "model__min_child_weight": [1, 3, 5],

    "model__subsample": [0.6, 0.8, 1.0],

    "model__colsample_bytree": [0.6, 0.8, 1.0],

    "model__gamma": [0, 0.1, 0.3, 0.5],

    "model__reg_alpha": [0, 0.1, 1],

    "model__reg_lambda": [1, 3, 5]
}

In [141]:
random_search = RandomizedSearchCV(

    estimator=pipeline,

    param_distributions=param_grid,

    n_iter=25,

    scoring="f1",

    cv=5,

    verbose=2,

    n_jobs=-1,

    random_state=42
)

# Train

random_search.fit(X_train, y_train)

# Best parameters

print("Best Parameters:")
print(random_search.best_params_)

# Best score

print("Best F1 Score:")
print(random_search.best_score_)

# Best model

best_model = random_search.best_estimator_

# Predictions

y_pred = best_model.predict(X_test)

# Report

print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 25 candidates, totalling 125 fits
[CV] END model__colsample_bytree=0.6, model__gamma=0.5, model__learning_rate=0.03, model__max_depth=3, model__min_child_weight=1, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.6; total time=   0.1s
[CV] END model__colsample_bytree=0.6, model__gamma=0.5, model__learning_rate=0.03, model__max_depth=3, model__min_child_weight=1, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.6; total time=   0.1s
[CV] END model__colsample_bytree=0.6, model__gamma=0.5, model__learning_rate=0.03, model__max_depth=3, model__min_child_weight=1, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.6; total time=   0.1s
[CV] END model__colsample_bytree=0.6, model__gamma=0.5, model__learning_rate=0.03, model__max_depth=3, model__min_child_weight=1, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.6; total tim

In [142]:
confusion_matrix(y_test, y_pred)


array([[758, 278],
       [ 58, 315]])

In [146]:
cm = confusion_matrix(y_test, y_pred)

# Convert to DataFrame
cm_df = pd.DataFrame(
    cm,
    index=['Actual 0', 'Actual 1'],
    columns=['Predicted 0', 'Predicted 1']
)

# Extract values
TN = cm_df.iloc[0, 0]
FP = cm_df.iloc[0, 1]
FN = cm_df.iloc[1, 0]
TP = cm_df.iloc[1, 1]

# Calculate metrics
precision = TP / (TP + FP)
recall = TP / (TP + FN)

print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))


Precision: 0.531
Recall: 0.845
